In [51]:
import pptx
from pptx.enum.shapes import MSO_SHAPE_TYPE
import re

In [52]:
#------------------------
# 対象の pptx のインスタンス生成
#------------------------
# pptx の内容をPowerPointで書き換えた時はここから生成をやり直す！
#
ppt = pptx.Presentation('sample1.pptx')

In [44]:
#------------------------
# slide 取得
#------------------------

# pptx 内の slide のリスト取得
slides = ppt.slides  # pptx 内の slide のリスト取得

slide_index = 3
sld = slides[slide_index]  # list になっているので index で slide を取得できる

sld = ppt.slides[slide_index] # この書き方でも良い

In [45]:
#------------------------
# shape の取得と扱い方
#   - list の取得
#   - type のチェック
#   - shapeのテキストの取得
#------------------------
#
# sld.shapesで slide の中の shape のリストが取得できる。
#
for shape in sld.shapes:
    #
    # shapeは placeholderとそれ以外がある。
    # placeholde の場合は placeholder_format.type で種類がわかる
    #    placeholderのtypeは PP_PLACEHOLDER_TYPE で定義されている
    #        TITLE, HEADER, FOOTER, DATE など
    # それ以外は shape_type で種類がわかる
    #    MSO_SHAPE_TYPE で定義されている
    #        TEXT_BOX, GROUP, PICTURE, CHART, TABLE など
    print()
    if shape.is_placeholder:
        # placeholder type は PP_PLACEHOLDER_TYPE
        print(f">>>place holder:type ---  {shape.placeholder_format.type} ---")
        
        #
        # テキストを持つオブジェクトはtext_frameを持っている。
        # shape.text でテキストを取得できる。
        #
        if shape.has_text_frame:
             print(shape.text)
        else:
             print("--テキスト無し--")
    else:
        print(f">>>NON place holder:shape type {shape.shape_type}")
        # shape_type は MSO_SHAPE_TYPE
        if shape.has_text_frame:
            # shapeのxmlを表示
            #print(shape.element.xml)
            print(shape.text)               
        else:
            print("--テキスト無し--")


>>>place holder:type ---  TITLE (1) ---
タイトル

>>>place holder:type ---  OBJECT (7) ---
テキスト

>>>place holder:type ---  FOOTER (15) ---
footer

>>>place holder:type ---  SLIDE_NUMBER (13) ---
4

>>>NON place holder:shape type TEXT_BOX (17)
Standalone Text 3

>>>NON place holder:shape type GROUP (6)
--テキスト無し--


In [46]:
#-------------------------
# テキストの行の処理
#-------------------------
# textframe の paragraphs で行のリストを取得できるので、行単位の操作も可能。
# inndent level など、行単位の属性を扱う時に使う。
#
for shape in sld.shapes:
    
    if shape.is_placeholder:
        print(f">>>shape type {shape.shape_type}/{shape.placeholder_format.type}")
    else:
        print(f">>>shape type {shape.shape_type}")
        
    if shape.has_text_frame:
        text_frame = shape.text_frame
        for paragraph in text_frame.paragraphs:
            if paragraph.text=="":
                print ("--(空白行)--")
            else:
                print (paragraph.text)
            print (f'   (indent level={paragraph.level})')
    else:
        print("--テキスト無し--")

>>>shape type PLACEHOLDER (14)/TITLE (1)
タイトル
   (indent level=0)
>>>shape type PLACEHOLDER (14)/OBJECT (7)
テキスト
   (indent level=0)
>>>shape type PLACEHOLDER (14)/FOOTER (15)
footer
   (indent level=0)
>>>shape type PLACEHOLDER (14)/SLIDE_NUMBER (13)
4
   (indent level=0)
>>>shape type TEXT_BOX (17)
Standalone Text 3
   (indent level=0)
>>>shape type GROUP (6)
--テキスト無し--


In [42]:
#------------------------
# xml直接参照の適用例
#------------------------
#
# 適用例１：代替テキストを xml から取り出す
#
def outputAltText(shape):
    #
    # テキストがあるオブジェクトにも代替テキストは定義できるが、ここでは、出力が見やすいように
    # テキストがない(=イメージなど）に限定する。
    #
    if not shape.has_text_frame:
        # 次行で xml を表示すると descr=... の定義が確認できる
        #print(shape.element.xml)
        xmlstr = shape.element.xml
        pattern = r'<p:cNvPr id=.*descr="(.*?)"'
      #  pattern = r'descr="(.*?)"'
        descrArray=re.findall(pattern,xmlstr)
      #  print(descrArray)
        if len(descrArray)==0:
            print("--代替テキスト無し--")
        else:
            # 文字列そのまま
            #descr=descrArray[0]
            # 文字にエンコードされた改行を変換
            descr=descrArray[0].replace("&#10;","\n")
            # 文字にエンコードされた改行を削除
            #descr=descrArray[0].replace("&#10;","")
            print(descr)

In [43]:
#  xml の直接参照
#-------------------
# 全ての属性を python-pptx がサポートしているわけでは無い。
# サポートしていない属性を見たい時など xml を直接みる事もできる。
# xml への到達の仕方は色々。
#
# !!!
# !!! 以下のサンプルでは xml の print はコメントを外さないと出ないようになっている。 !!!
# !!! 出力が長くなりすぎるので。
# !!!
#
for shape in sld.shapes:
    print()
    if shape.is_placeholder:        
        # placeholder type is defined by PP_PLACEHOLDER_TYPE
        print(f">>>place holder:type ---  {shape.placeholder_format.type} ---")
        
        # コメントを外すと、shape共通のxmlを表示
        #print('--- shape共通のxmlを表示 ---')
        #print(shape.element.xml)
        
        # コメントを外すと、placeholder特有の属性のxmlを表示
        #print('--- placeholder特有の属性のxmlを表示 ---')
        #print(shape.placeholder_format.element.xml)
        
        print()
        if shape.has_text_frame:
             print(shape.text)
        else:
             print("--テキスト無し--")
    else:
        print(f">>>NON place holder:shape type {shape.shape_type}")
        # shape_type is defined by MSO_SHAPE_TYPE
        
        # コメントを外すと、shape共通のxmlを表示
        #print('--- shape共通のxmlを表示 ---')
        #print(shape.element.xml)
        
        print()
        if shape.has_text_frame:
            print(shape.text)

            #
            # 行(Paragraph)の属性は _pPr を使ってアクセスできる。
            # _pPr は内部実装のPragraph Propertyなのでハッキングした手法だが
            #  調べると、結構、このやり方が出てくる。
            #
            for paragraph in text_frame.paragraphs:
                if paragraph.text=="":
                    pass
                else:
                    print (paragraph.text)
                    pPr = paragraph._pPr
                    # コメントを外すと、Property属性のxmlを表示
                    #print(pPr.xml)
                    # 例えば、以下のように番号リストの定義があるかどうかを確認できる
                    an = pPr.find(qn('a:buAutoNum'))
                    if an is None:
                        print("-- 番号リストではない --")
                    else:
                        print("-- 番号リスト --")
        
        else:
             print("--テキスト無し--")

        outputAltText(shape)



>>>place holder:type ---  TITLE (1) ---

タイトル

>>>place holder:type ---  OBJECT (7) ---

テキストエリアに書かれている文字です
これは2行目

>>>place holder:type ---  FOOTER (15) ---

footer

>>>place holder:type ---  SLIDE_NUMBER (13) ---

3

>>>NON place holder:shape type PICTURE (13)

--テキスト無し--
地図です！
場所はIBM箱崎事業所。
この代替テキストはこの行を含めて3行あります。

>>>NON place holder:shape type PICTURE (13)

--テキスト無し--
--代替テキスト無し--

>>>NON place holder:shape type AUTO_SHAPE (1)


テキスト欄４　（テキスト欄１のコピー）
-- 番号リストではない --
テキスト欄４　2行目　インデント　bulletあり
-- 番号リストではない --
テキスト欄４　3行目　インデント bulletなし
-- 番号リストではない --
テキスト欄４ 4行目　インデントなし(1行目と同列)　番号で箇条書き
-- 番号リスト --

>>>NON place holder:shape type AUTO_SHAPE (1)


テキスト欄４　（テキスト欄１のコピー）
-- 番号リストではない --
テキスト欄４　2行目　インデント　bulletあり
-- 番号リストではない --
テキスト欄４　3行目　インデント bulletなし
-- 番号リストではない --
テキスト欄４ 4行目　インデントなし(1行目と同列)　番号で箇条書き
-- 番号リスト --

>>>NON place holder:shape type PICTURE (13)

--テキスト無し--
3D メガネ 単色塗りつぶし

>>>NON place holder:shape type PICTURE (13)

--テキスト無し--
ドングリ 単色塗りつぶし

>>>NON place holder:shape type CHART

In [49]:
#-------------------
# GroupShapeの処理
#-------------------
# パワーポイントでグループ化すると、複数個のshapeが１つのグループ(GroupShape)になる。
# それぞれのshapeの操作はグループに所属するchild shapeを取得して行う。
# g_shapes がGroupShapeの親インスタンスだったとして、
#    g_shapes.shapes で、child shape のリストが取得できる。

#
# 以下は、リカーシブルなファンクションを使って親子関係をたどり、shapeのテキストを出力する例。
#
def printGroup(g_shapes):
    for g_shape in g_shapes.shapes:
        if g_shape.shape_type == MSO_SHAPE_TYPE.GROUP:
            printGroup(g_shape)
        else:
            if g_shape.has_text_frame:
                print(g_shape.text)
for shape in sld.shapes:
    if not shape.is_placeholder:
        if shape.shape_type == MSO_SHAPE_TYPE.GROUP:
            printGroup(shape)
            #for groupshape in shape.shapes:
            #    if groupshape.has_text_frame:
            #        print(groupshape.text)


Group Text 5
Group Text 1
Group Text 2
Group Text 4
Group2 Text 2-1
Group2 Text 2-2
Group2 Text 2-3


In [53]:
#-----------------------
#スピーカーノート取得
#-----------------------
if sld.has_notes_slide:
    print('--Speaker note is defined')
    notes_text = sld.notes_slide.notes_text_frame.text
    if notes_text:
        # テキストがスペースだけというケースもあり得る
        if notes_text.strip():
            print(f'"{notes_text}"')
        else:
            print('テキストはあるがスペースだけ')
    else:
        print('notes_slideはあるがテキストがない')
        # notes_slideのインスタンスはあるが、テキストのインスタンスがない状態。
        # pptのUI上でNoteのエリアにカーソルを移しただけでセーブするとこの状態になる。
        # 一度、入力した情報を全て削除した場合もこの状態。
else:
    print('--Speaker note is NOT defined')
    # ノートがない時は notes_slide インスタンスが新規に生成されている。以下で確認できる。
    if sld.notes_slide is not None:
        print('notes_slideのインスタンスが新規に生成されている')
    else:
        # ここを通ることはない
        print('notes_slideのインスタンスがない')

--Speaker note is defined
"　　ggg  "
